In [ ]:
import os, sys, pathlib

ROOT = pathlib.Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

for _m in [m for m in sys.modules if m == 'scripts' or m.startswith('scripts.')
           or m == 'src' or m.startswith('src.')]:
    del sys.modules[_m]

import matplotlib.pyplot as plt

CONDITIONS = ('baseline_100', 'phase_noise_100', 'iq_imbalance_100',
              'quantization_100', 'all_100')
BASELINE = 'baseline_100'
print('repo root:', ROOT)

In [ ]:
# Inference only: fills in the 25 (train, eval) cells that are not already on disk.
# The diagonal comes from src.predict.run_all and is left alone.
from scripts.compare_all_conditions import score_matrix

SCORE = False

if SCORE:
    score_matrix(CONDITIONS)
else:
    print('skipped; set SCORE = True to write the missing cross-domain predictions')

In [ ]:
from scripts.compare_all_conditions import check_guards, load_matrix, print_datasets

matrix = load_matrix(CONDITIONS, baseline=BASELINE)
print_datasets(matrix)
print()
check_guards(matrix)

In [ ]:
# Part 1 -- what each impairment costs, paired by seed against the clean baseline.
from scripts.compare_all_conditions import plot_deltas, print_summary

rows = print_summary(matrix)
plot_deltas(rows, BASELINE)
plt.show()

In [ ]:
# Part 2 -- every model on every condition's test split.
from scripts.compare_all_conditions import (
    cross_domain_matrix, off_diagonal_cost, plot_matrix, print_cross_domain)

values = print_cross_domain(matrix)

plot_matrix(values, matrix.conditions,
            f'macro accuracy at SNR >= 0 dB, median over {len(matrix.seeds)} seeds',
            'macro accuracy (%)')
plot_matrix(off_diagonal_cost(values), matrix.conditions,
            "cell minus its row's diagonal: cost of leaving the training domain",
            'delta macro accuracy (pp)', diverging=True)
plt.show()